# TP Séance 1 — Comparaison de modèles open-source (exécution locale sur Colab)

**Objectif :** sur un même jeu de prompts, faire répondre plusieurs LLM ouverts et comparer leurs réponses (exactitude, clarté, créativité, rapidité).

**Outils :** Google Colab (GPU gratuit), `transformers`. Aucun token ni crédit n'est nécessaire, les modèles sont téléchargés et exécutés directement sur la machine Colab.

**Avant de commencer :** `Exécution > Modifier le type d'exécution > GPU (T4)`.

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", DEVICE)
if DEVICE == "cpu":
    print("Activez le GPU : Exécution > Modifier le type d'exécution > GPU (sinon le TP sera très lent).")

## 1. Configuration

Modèles ouverts et non « gated » (aucune acceptation de licence requise). Les trois prompts reprennent ceux du TP initial.

In [ ]:
MODELS = {
    "Qwen 2.5 1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen 2.5 3B": "Qwen/Qwen2.5-3B-Instruct",
    "Phi 3.5 mini": "microsoft/Phi-3.5-mini-instruct",
}

PROMPTS = {
    "Factuel": "Quelle est la capitale de l'Australie et depuis quand le est-elle ?",
    "Créatif": "Écris un slogan pour une application mobile qui aide les étudiants à réviser leurs examens.",
    "Raisonnement": "Un train part de Grenoble à 8h00 à 90 km/h. Un second part de Lyon (110 km plus loin) "
    "à 8h15 à 110 km/h en direction de Grenoble. À quelle heure se croisent-ils ?",
}

MAX_NEW_TOKENS = 300

## 2. Fonctions d'inférence

In [ ]:
import time
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM


def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer, model


def ask_model(tokenizer, model, prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = round(time.time() - start, 2)

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), elapsed

## 3. Comparaison des modèles

Chaque modèle est chargé, interrogé sur les 3 prompts, puis libéré de la mémoire GPU avant de passer au suivant.

In [ ]:
results = []

for model_name, model_id in MODELS.items():
    print(f"Chargement de {model_name} ({model_id})...")
    try:
        tokenizer, model = load_model(model_id)
    except Exception as e:
        results.append({"Modèle": model_name, "Prompt": "-", "Temps (s)": None, "Réponse": f"[Erreur de chargement : {type(e).__name__}: {str(e)[:200]}]"})
        continue

    for prompt_name, prompt in PROMPTS.items():
        answer, elapsed = ask_model(tokenizer, model, prompt)
        results.append({"Modèle": model_name, "Prompt": prompt_name, "Temps (s)": elapsed, "Réponse": answer})

    del model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("Terminé.")

In [ ]:
df = pd.DataFrame(results)
df.to_csv("resultats_modeles.csv", index=False)
df

## 4. Lecture comparative

Lisez les réponses prompt par prompt pour comparer les modèles côte à côte.

In [ ]:
for prompt_name in PROMPTS:
    print(f"===== {prompt_name} =====")
    for _, row in df[df["Prompt"] == prompt_name].iterrows():
        print(f"\n--- {row['Modèle']} ({row['Temps (s)']} s) ---")
        print(row["Réponse"])
    print()

## 5. Grille d'observation

Notez chaque modèle de 1 à 5 sur chaque critère (à compléter après lecture des réponses).

| Modèle | Prompt | Exactitude | Clarté | Créativité | Rapidité (temps) | Commentaire |
|---|---|---|---|---|---|---|
| Qwen 2.5 1.5B | Factuel | | | | | |
| Qwen 2.5 3B | Factuel | | | | | |
| Phi 3.5 mini | Factuel | | | | | |
